In [ ]:
# Prérempli. Copiez et exécutez simplement cette cellule.
import os, math, re, random
from glob import glob
from pathlib import Path

# Chemins - à modifier si nécessaire.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data" / "cats_dogs").exists():
    NOTEBOOK_DIR = Path(r"D:\FORMATION TTA\DI-BOOTCAMP\Week5\Day5\ExerciseXP")

MPLCONFIGDIR = NOTEBOOK_DIR / ".matplotlib"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42)
tf.random.set_seed(42)

DATA_ROOT = NOTEBOOK_DIR / "data" / "cats_dogs"
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir = (DATA_ROOT / "test" / "test") if (DATA_ROOT / "test" / "test").exists() else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# Construction des DataFrames à partir des dossiers.
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")
    files = []
    for ex in exts:
        files.extend(glob(str(folder / "**" / ex), recursive=True))

    if not files:
        raise FileNotFoundError(f"Aucune image trouvée dans {folder}")

    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat", "cats"}:
                label = "cat"
            elif parent in {"dog", "dogs"}:
                label = "dog"
            elif re.search(r"(^|[^a-z])cat([^a-z]|$)", name):
                label = "cat"
            elif re.search(r"(^|[^a-z])dog([^a-z]|$)", name):
                label = "dog"
            else:
                continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})

    df = pd.DataFrame(rows)
    if labeled and df.empty:
        raise ValueError(f"Des images ont été trouvées dans {folder}, mais aucune étiquette cat/dog n'a pu être déduite.")
    return df

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full = build_df_from_folder(test_dir, labeled=False)

# Séparation entraînement/validation.
from sklearn.model_selection import train_test_split

df_tr, df_val = train_test_split(
    df_train_full,
    test_size=0.2,
    stratify=df_train_full["label"],
    random_state=seed
)

# Générateurs d'images.
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    batch_size=batch_size,
    shuffle=True,
    seed=seed,
    validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val,
    x_col="filepath",
    y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary",
    batch_size=batch_size,
    shuffle=False,
    validate_filenames=False
)

# Jeu de test sans étiquettes, utilisé uniquement pour l'inférence.
test_flow = test_gen.flow_from_dataframe(
    df_test_full,
    x_col="filepath",
    y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    batch_size=batch_size,
    shuffle=False,
    validate_filenames=False
)

print({
    "train": train_flow.samples,
    "val": val_flow.samples,
    "test": test_flow.samples,
    "class_indices": train_flow.class_indices,
})


Le jeu de données contient deux classes : chat et chien. Le nombre exact d'images par classe peut être obtenu à partir de `train_flow.labels` et de `train_flow.class_indices`. Les classes sont généralement équilibrées dans ce jeu de données, ce qui limite le risque de biais d'apprentissage.

Les images présentent une forte variabilité visuelle : différentes poses des animaux, variations d'échelle, conditions d'éclairage diverses, arrière-plans complexes et races multiples. Cette variabilité justifie l'utilisation de techniques d'augmentation de données afin d'améliorer la capacité de généralisation du modèle.


In [ ]:
images, labels = next(train_flow)

plt.figure(figsize=(10,10))

for i in range(9):
    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    label = "dog" if labels[i] == 1 else "cat"

    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()

Indices visuels

Le modèle peut distinguer les chats des chiens grâce à plusieurs indices visuels :
- la forme des oreilles ;
- la longueur du museau ;
- la texture du pelage ;
- les proportions du visage ;
- la taille relative du corps ;
- certaines caractéristiques spécifiques aux races.


3. Définir l'architecture du modèle

Description rédigée

Le modèle CNN proposé comporte trois blocs convolutionnels. Chaque bloc utilise une couche `Conv2D` suivie d'une couche `MaxPooling2D` afin de réduire progressivement les dimensions spatiales tout en conservant les caractéristiques importantes.

Un taux de `Dropout` est ajouté après les couches convolutionnelles et denses pour limiter le surapprentissage, en empêchant le réseau de dépendre excessivement de certains neurones.

Après les couches convolutionnelles, les cartes de caractéristiques sont aplaties à l'aide d'une couche `Flatten`. Deux couches `Dense` sont ensuite utilisées pour apprendre des représentations plus complexes. La couche de sortie contient une seule unité avec une activation sigmoïde, adaptée à une classification binaire.


In [ ]:
# Architecture CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, Input, MaxPooling2D

model = Sequential([
    Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D(2, 2),

    Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D(2, 2),

    Dropout(0.25),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()


4. Configuration de l'optimisation

Réponse rédigée

L'optimiseur Adam est choisi car il converge rapidement et nécessite peu d'ajustements manuels.

Le taux d'apprentissage initial est fixé à 0.001, une valeur couramment utilisée qui offre un bon compromis entre vitesse d'apprentissage et stabilité.

Une taille de lot de 32 est utilisée car elle convient à la plupart des GPU et limite la consommation de mémoire.

L'arrêt précoce est activé pour interrompre l'entraînement lorsque la perte de validation cesse de s'améliorer. Un planificateur de taux d'apprentissage peut également être utilisé pour réduire automatiquement le taux d'apprentissage lorsque la validation stagne.


In [ ]:
# Callbacks et poids des classes
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3
)

classes = np.unique(train_flow.labels)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_flow.labels
)
class_weights = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

print(class_weights)



In [ ]:
# 5. Entraîner le modèle
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=25,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights
)


In [ ]:
# Courbes
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['train','validation'])
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['train','validation'])
plt.title("Loss")

plt.show()

Interprétation du surapprentissage

Le surapprentissage apparaît lorsque la précision d'entraînement continue d'augmenter alors que la précision de validation stagne ou diminue. On observe également une augmentation de la perte de validation tandis que la perte d'entraînement diminue.

Pour réduire ce phénomène, nous utilisons l'augmentation de données, le `Dropout` et l'arrêt précoce.


In [ ]:
# 6. Évaluation
from sklearn.metrics import classification_report, confusion_matrix

val_probs = model.predict(val_flow).ravel()
val_preds = (val_probs > 0.5).astype(int)

cm = confusion_matrix(val_flow.labels, val_preds, labels=[0, 1])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["cat", "dog"],
    yticklabels=["cat", "dog"]
)
plt.xlabel("Prédiction")
plt.ylabel("Vraie classe")
plt.show()

print(
    classification_report(
        val_flow.labels,
        val_preds,
        labels=[0, 1],
        target_names=["cat", "dog"]
    )
)


Interprétation

La matrice de confusion permet d'identifier les faux positifs et les faux négatifs. Si de nombreux chats sont classés comme chiens, cela peut indiquer que certaines caractéristiques visuelles communes sont mal apprises par le modèle.

Le seuil de 0.5 peut être ajusté afin d'optimiser soit la précision, soit le rappel, selon les besoins du projet.


In [ ]:
# 7. Inférence sur le test
probs = model.predict(test_flow).ravel()
pred_labels = np.where(probs > 0.5, "dog", "cat")

submission = pd.DataFrame({
    "filepath": test_flow.filepaths,
    "prob_dog": probs,
    "pred_label": pred_labels
})

submission.to_csv(NOTEBOOK_DIR / "predictions.csv", index=False)
submission.head()


Vérification manuelle

Pour contrôler la cohérence des prédictions, on sélectionne aléatoirement plusieurs images du fichier CSV, puis on compare visuellement l'image et la prédiction du modèle.


8. Comparaison : modèle de référence vs augmentation

Réponse

Le modèle utilisant l'augmentation des données obtient généralement une meilleure précision de validation et un écart plus faible entre l'entraînement et la validation.

Le modèle sans augmentation présente souvent un surapprentissage plus important, car il voit toujours les mêmes images pendant l'entraînement.


In [ ]:
# 9. Gestion du déséquilibre
# Les poids des classes ont été calculés avant l'entraînement et passés à model.fit.
class_weights


In [ ]:
# 10. Sauvegarde
import json

model.save(NOTEBOOK_DIR / "cats_dogs_model.keras")

config = {
    "optimizer": "adam",
    "batch_size": batch_size,
    "img_size": [IMG_HEIGHT, IMG_WIDTH],
    "class_indices": train_flow.class_indices
}

with open(NOTEBOOK_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)


Explication

La sauvegarde des poids permet de réutiliser le modèle entraîné. La sauvegarde des métadonnées garantit la reproductibilité en conservant les paramètres d'entraînement, l'architecture et les hyperparamètres utilisés.


11. Extension proposée

Je recommande l'utilisation du transfert d'apprentissage avec `MobileNetV2`.

Cette architecture a déjà appris des caractéristiques générales sur ImageNet. Elle permet généralement :
- une convergence plus rapide ;
- une meilleure précision ;
- moins de données nécessaires ;
- une réduction du temps d'entraînement.
